# HR Support Ticket RAG Setup

This notebook creates a ChromaDB collection from the `data/support_ticket.txt` file so we can answer employee HR support questions from a local knowledge base.

In [1]:
import re
from pathlib import Path
from typing import Dict, List

import chromadb

repo_root = Path.cwd().resolve()
if repo_root.name == 'rag_setup':
    repo_root = repo_root.parent

print(f'Project root: {repo_root}')


Project root: /workspaces/Ai-Agent-course


In [2]:
def parse_qa_pairs(file_path: str) -> List[Dict[str, str]]:
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()

    pairs = [part.strip() for part in re.split(r'\n\s*\n', content) if part.strip()]
    qa_pairs: List[Dict[str, str]] = []

    for idx, pair in enumerate(pairs):
        question = ''
        answer = ''

        for line in pair.splitlines():
            if line.startswith('Question:'):
                question = line.replace('Question:', '', 1).strip()
            elif line.startswith('Answer:'):
                answer = line.replace('Answer:', '', 1).strip()

        if question and answer:
            qa_pairs.append({'question': question, 'answer': answer, 'id': f'hr_qa_{idx}'})

    return qa_pairs

support_file = repo_root / 'data' / 'support_ticket.txt'
qa_pairs = parse_qa_pairs(str(support_file))
print(f'Total HR Q&A entries found: {len(qa_pairs)}')
print(qa_pairs[0]['question'])


Total HR Q&A entries found: 213
How do I request annual leave from HR?


In [3]:
def prepare_hr_support_documents(file_path: str) -> Dict[str, List[str]]:
    qa_pairs = parse_qa_pairs(file_path)

    documents = []
    metadatas = []
    ids = []

    for qa in qa_pairs:
        document_text = (
            f"Question: {qa['question']}\n"
            f"Answer: {qa['answer']}\n\n"
            'This support ticket entry provides HR guidance for employees.'
        )

        keywords = re.findall(r'\b\w+\b', (qa['question'] + ' ' + qa['answer']).lower())

        metadata = {
            'question': qa['question'],
            'answer': qa['answer'],
            'question_length': len(qa['question']),
            'answer_length': len(qa['answer']),
            'keywords': ' '.join(sorted(set(keywords))),
            'topic': 'hr_support_ticket',
        }

        documents.append(document_text)
        metadatas.append(metadata)
        ids.append(qa['id'])

    return {'documents': documents, 'metadatas': metadatas, 'ids': ids}

support_docs = prepare_hr_support_documents(str(support_file))
print(f"Prepared {len(support_docs['documents'])} documents")


Prepared 213 documents


In [4]:
def setup_support_ticket_chromadb(file_path: str, collection_name: str = 'hr_support_qna'):
    chroma_path = repo_root / 'chroma'
    client = chromadb.PersistentClient(str(chroma_path))

    try:
        client.delete_collection(collection_name)
    except Exception:
        pass

    collection = client.create_collection(
        name=collection_name,
        metadata={'description': 'HR support ticket Q&A knowledge base'}
    )

    data = prepare_hr_support_documents(file_path)
    collection.add(documents=data['documents'], metadatas=data['metadatas'], ids=data['ids'])

    print(f"Added {len(data['documents'])} HR support Q&A entries to '{collection_name}'")
    return collection

hr_support_db = setup_support_ticket_chromadb(str(support_file), 'hr_support_qna')
print(hr_support_db.count())


Added 213 HR support Q&A entries to 'hr_support_qna'
213


In [5]:
query = 'How do I request annual leave from HR?'
results = hr_support_db.query(query_texts=[query], n_results=2)
for doc in results['documents'][0]:
    print(doc)
    print('---')


Question: How do I request annual leave from HR?
Answer: Submit a leave request through the HR portal or manager approval workflow. Include your leave dates, reason, and make sure your remaining leave balance is checked before approval.

This support ticket entry provides HR guidance for employees.
---
Question: How do I request annual leave after a promotion or role change?
Answer: Submit the leave request as usual, and HR can confirm whether your leave balance or policy applies differently after the role change.

This support ticket entry provides HR guidance for employees.
---
